# Notebook 4 — Adding Structured Business Data

### Build an AI Business Copilot 🏔️ · Session 2

Welcome to evening two! In Session 1 you built a copilot that answers **policy** questions
from documents (RAG). But customers also ask questions no document can answer:

- *"Where is my order #1001?"*
- *"Is the Voyager 65L Backpack in stock?"*

Those answers live in Trailhead's **databases** — orders, shipments, inventory — and they
change every day. In this notebook we teach the copilot to **look up real business
records** using a technique called **tool use** (also known as *function calling*).

By the end you'll be able to:

1. Load Trailhead's business data (CSV tables) with pandas.
2. Explain why we give Claude **tools** instead of pasting the whole database in.
3. Write simple lookup functions and describe them to Claude as tools.
4. Run the **tool-use loop** so the copilot answers account questions from real data.

> 💸 Tool use makes ~2 short Claude calls per question. Still Haiku 4.5 — a fraction of a
> cent each.

## 1. Setup

This notebook works with the data tables, so we don't need the embedding libraries — just
`anthropic` and `pandas`. Run the three setup cells.

In [ ]:
%pip install -q anthropic pandas

In [ ]:
# --- Make the workshop files available (Colab, Databricks, or local) ---
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/YOUR-ORG/trailhead-copilot"   # <-- ✏️ set to the real repo URL

def find_repo_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if all((base / d).is_dir() for d in ("data", "documents", "src")):
            return base
    cloned = Path.cwd() / "trailhead-copilot"
    return cloned if (cloned / "src").is_dir() else None

root = find_repo_root()
if root is None:
    print("Cloning the workshop repo...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    root = find_repo_root()
if root is None:
    raise RuntimeError("Workshop files not found — set REPO_URL to the real repository URL.")

sys.path.insert(0, str(root / "src"))
print("Using workshop files at:", root)

In [ ]:
import os, getpass
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Paste your Anthropic API key: ")
print("API key is set. ✅")

## 2. Configuration

In [ ]:
MODEL = "claude-haiku-4-5"
MAX_TOKENS = 500

## 3. Load Trailhead's business data

Trailhead's records are stored as CSV tables (in a real company these would be database
tables — same idea). `trailhead.load_data()` loads them all into pandas **DataFrames**, one
per table.

In [ ]:
import trailhead as th
import pandas as pd

data = th.load_data()
print("Tables:", list(data.keys()))

# Pull out the tables we'll use as regular variables.
orders = data["orders"]
shipments = data["shipments"]
products = data["products"]
inventory = data["inventory"]

orders.head()

In [ ]:
# The shipment details for one order:
shipments[shipments["order_id"] == 1001]

## 4. Why not just paste the whole database into the prompt?

We *could* dump every order into the prompt and let Claude read it. But that's a bad idea:

- 📦 **Too big.** Thousands of orders won't fit — and you'd pay for every token, every time.
- 🕰️ **Always changing.** Statuses and stock update constantly; a pasted snapshot is stale
  the moment you send it.
- 🔒 **Insecure.** You don't want to expose your entire customer database to answer one
  question.

Instead, we give Claude **tools**: small functions it can *call* to fetch exactly the one
record it needs, right when it needs it.

> 🤝 **A helpful analogy.** Think of Claude as a support agent who can't see the database
> directly, but can ask a colleague to look things up. Claude says *"please get me the
> status of order 1001"*; our code runs the lookup and hands back the answer; Claude writes
> the reply. **You** stay in control of what data is accessible.

## 5. Write the lookup functions

These are just ordinary Python functions over the DataFrames — nothing AI about them yet.
Each returns a small dictionary of facts. We handle the "not found" case so the copilot can
respond gracefully.

In [ ]:
def clean(value):
    # pandas uses NaN for empty cells; turn that into a clean None.
    return None if pd.isna(value) else value

def get_order_status(order_id):
    order_id = int(order_id)
    order = orders[orders["order_id"] == order_id]
    if order.empty:
        return {"error": "No order found with ID " + str(order_id)}
    o = order.iloc[0]
    result = {
        "order_id": order_id,
        "order_status": clean(o["status"]),
        "shipping_method": clean(o["shipping_method"]),
        "order_total": float(o["total"]),
    }
    ship = shipments[shipments["order_id"] == order_id]
    if not ship.empty:
        s = ship.iloc[0]
        result["shipment_status"] = clean(s["status"])
        result["carrier"] = clean(s["carrier"])
        result["tracking_number"] = clean(s["tracking_number"])
        result["estimated_delivery"] = clean(s["estimated_delivery"])
        result["delivered_date"] = clean(s["delivered_date"])
    return result

def check_inventory(product_name):
    matches = products[products["name"].str.contains(product_name, case=False, na=False)]
    if matches.empty:
        return {"error": "No product matching '" + product_name + "'."}
    found = []
    for _, p in matches.iterrows():
        inv = inventory[inventory["product_id"] == p["product_id"]]
        units = int(inv.iloc[0]["units_in_stock"]) if not inv.empty else 0
        found.append({"product": p["name"], "units_in_stock": units, "in_stock": units > 0})
    return {"matches": found}

# Test them directly — no Claude involved yet:
print(get_order_status(1001))
print(check_inventory("Voyager"))

## 6. Describe the tools to Claude

Claude can't see your Python functions — you have to *describe* them so it knows what each
one does and what input it needs. This description is a small piece of JSON: a **name**, a
plain-English **description**, and an **input schema** (the parameters).

> 📝 The `description` really matters — it's how Claude decides *which* tool to use and
> *when*. Write it like instructions to a new teammate.

In [ ]:
tools = [
    {
        "name": "get_order_status",
        "description": (
            "Look up the status, shipping method, carrier, tracking number, and estimated "
            "delivery date for a customer order, given its numeric order ID."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "integer", "description": "The numeric order ID, e.g. 1001"}
            },
            "required": ["order_id"],
        },
    },
    {
        "name": "check_inventory",
        "description": (
            "Check how many units of a product are currently in stock. Accepts a full or "
            "partial product name."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "product_name": {"type": "string", "description": "Full or partial product name, e.g. 'Voyager 65L'"}
            },
            "required": ["product_name"],
        },
    },
]

# A lookup table so we can call the right Python function by name.
TOOL_FUNCTIONS = {
    "get_order_status": get_order_status,
    "check_inventory": check_inventory,
}
print("Described", len(tools), "tools to Claude.")

## 7. Peek: what does Claude do with a tool?

Let's make **one** call with the tools attached and see what comes back. Watch the
`stop_reason` — instead of answering, Claude will **ask us to run a tool**.

In [ ]:
import anthropic
client = anthropic.Anthropic()

SYSTEM = (
    "You are a support assistant for Trailhead Supply Co. Use the available tools to look "
    "up real order and inventory data before answering. Only state facts returned by a "
    "tool - never guess an order status or stock level. If a tool reports no matching "
    "record, tell the customer politely and suggest contacting support@trailheadsupply.example. "
    "Keep replies brief."
)

peek = client.messages.create(
    model=MODEL, max_tokens=MAX_TOKENS, system=SYSTEM, tools=tools,
    messages=[{"role": "user", "content": "Where is order #1001?"}],
)

print("stop_reason:", peek.stop_reason, "\n")
for block in peek.content:
    if block.type == "text":
        print("Claude says:", block.text)
    elif block.type == "tool_use":
        print("Claude wants to call tool:", block.name)
        print("           with input:", block.input)

See it? `stop_reason` is **`tool_use`**, and Claude has requested `get_order_status` with
`order_id = 1001`. It hasn't answered yet — it's waiting for us to run the tool and hand
back the result. That back-and-forth is the **tool-use loop**.

## 8. Run the tool-use loop

The loop is simple — repeat until Claude stops asking for tools:

1. Send the conversation to Claude.
2. If Claude asked for a tool (`stop_reason == "tool_use"`): run the matching Python
   function, send the result back, and go again.
3. Otherwise, Claude has its final answer — return it.

The function below does exactly that, printing each tool call so you can watch it work.

In [ ]:
import json

def run_with_tools(user_message, verbose=True):
    messages = [{"role": "user", "content": user_message}]
    while True:
        response = client.messages.create(
            model=MODEL, max_tokens=MAX_TOKENS, system=SYSTEM,
            tools=tools, messages=messages,
        )

        # No tool requested -> Claude's final answer.
        if response.stop_reason != "tool_use":
            return "".join(b.text for b in response.content if b.type == "text")

        # Otherwise, run every tool Claude asked for and feed the results back.
        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = TOOL_FUNCTIONS[block.name](**block.input)
                if verbose:
                    print(f"  🔧 {block.name}({block.input}) -> {result}")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(result),
                })
        messages.append({"role": "user", "content": tool_results})

# Try it end to end:
print("ANSWER:", run_with_tools("Where is order #1001?"))

🎉 That's the copilot answering from **live data**. Read the trace: Claude called
`get_order_status(1001)`, our function returned the real record, and Claude turned it into a
friendly reply — grounded in facts, not guessed.

## 9. More questions — and graceful failure

The same loop handles inventory questions, and it degrades gracefully when a record doesn't
exist (thanks to the `"error"` our functions return).

In [ ]:
print("Q1:", run_with_tools("Do you have the Voyager 65L Backpack in stock?"))
print()
print("Q2:", run_with_tools("What's the status of order 9999?"))   # no such order

For order 9999, the tool returned an error, and Claude relayed that politely instead of
inventing a status. **You** decided what the tool exposes and what happens on a miss — the
model just reports what it's told.

> ✏️ **Try it.** Ask your own account questions below. Mix order lookups and stock checks.
> Try a question that needs **two** lookups (e.g., an order status *and* whether a product
> is in stock) — Claude will call both tools.

In [ ]:
for q in [
    "Is order 1007 delivered yet?",
    "How many Traverse Mid Hiking Boots do you have?",
    "Where is order 1003, and do you have the Blaze stove in stock?",
]:
    print("Q:", q)
    print("A:", run_with_tools(q, verbose=False))
    print("-" * 70)

## 10. What we built, and what's next

Your copilot now has **two skills**:

- **Documents (RAG)** — policy questions (Notebooks 2–3).
- **Data (tool use)** — account questions like orders and inventory (this notebook).

But right now *you* decide which skill to use by choosing which function to call. A real
copilot gets a single question and has to figure that out itself.

**Notebook 5** adds a **router**: Claude reads the incoming question, decides whether it's a
*policy* question or a *data* question, and sends it to the right skill — combining
everything into one `ask_copilot()` entry point.

## 11. Recap

- Some answers live in **databases**, not documents — and they change constantly.
- **Tool use (function calling)** lets Claude *request* a lookup instead of us pasting data.
- You write plain Python functions, **describe** them to Claude, and run the **tool-use
  loop**: Claude asks → you run the tool → Claude answers.
- You control exactly what data is exposed — a safer, cheaper, always-current design.

## 12. Exercises

**Guided**
1. Ask about an order that is still **Processing** (try a few IDs from `orders`). How does
   Claude describe an order with no tracking yet? Is that a reasonable customer reply?
2. Change the `SYSTEM` prompt to tell the assistant to always end with *"Is there anything
   else I can help with?"*. Re-run a question.

**Challenge**
3. Write a **third tool**, `get_return_window(product_name)`, that returns a product's
   `return_window_days` from the `products` table. Add it to `tools` and `TOOL_FUNCTIONS`,
   then ask *"How long do I have to return the Daybreak 30L Daypack?"* (Notice this blends
   data *and* policy — a nice preview of routing.)
4. Improve `check_inventory` so that if a product's stock is at or below its `reorder_level`,
   the result includes `"low_stock": true`. Ask about a low-stock item and see if Claude
   mentions it.

## 13. Learn more

- **Anthropic — Tool use (function calling) overview:** <https://docs.claude.com/en/docs/agents-and-tools/tool-use/overview>
- **Anthropic — Implementing tool use:** <https://docs.claude.com/en/docs/agents-and-tools/tool-use/implement-tool-use>
- **Codecademy — Intro to Generative AI:** <https://www.codecademy.com/learn/intro-to-generative-ai>
- **pandas — 10 minutes to pandas:** <https://pandas.pydata.org/docs/user_guide/10min.html>

---
> ### 👩‍🏫 Instructor notes (remove before distributing to students)
>
> - **Timing (~55 min):** ~8 setup + load data, ~7 "why tools" (§4), ~10 functions (§5–6),
>   ~8 the peek (§7), ~12 the loop + examples (§8–9), remainder try-it/exercises.
> - **Tool use is the hardest concept in the workshop.** The §7 "peek" (seeing
>   `stop_reason: tool_use` and the requested call) is what makes the loop click — don't
>   skip it. The colleague analogy (§4) lands well; reuse it.
> - Emphasize **you control the tools**: the model can only touch data you exposed via a
>   function. Good moment to mention this is safer than letting an LLM write raw SQL.
> - Order **#1001** is Shipped (has carrier + tracking), so it's a rich example; **#9999**
>   demonstrates graceful failure. Both are deterministic from the generated data.
> - The multi-tool question in §9's try-it shows Claude chaining/parallelizing calls — a
>   good "wow," but if a learner's model answers in one call, that's fine too.
> - **Cost:** each question is ~2 Haiku calls; a full run is well under a cent per student.